# Baseline Experiment
- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
- **Owner:** Nouroz Ali
---

Starter notebook: shared data loading and batch verification are provided; model implementation, training, and results remain to be completed. No results are fabricated.

Reuse the prepared Enron partitions and training vocabulary. Select configurations on validation data and reserve test evaluation for frozen final models.

## 1. Initial Configuration

### 1.1. Imports, Paths, and Device

In [3]:
# Import required libraries
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

# Locate the repository from either its root or the notebooks folder
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'src').is_dir() else current_dir.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Place this notebook in the repository notebooks folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import shared project modules
from src.config import BATCH_SIZE, TRAINING_SEEDS
from src.data import load_prepared_data
from src.reproducibility import set_seed
from src.text import EncodedTextDataset, make_data_loader

# Use CUDA when it is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Print the runtime and selected device
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.10
Interpreter: c:\Users\dancm\OneDrive\Documentos\humber-aigc-postgraduate\aigc-5500-advanced-deep-learning\midterm-project\AIGC5500-Midterm-Project\.venv\Scripts\python.exe
PyTorch: 2.9.1+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### 1.2. Experiment Configuration

In [ ]:
# Name this experiment and its owner
MODEL_NAME = 'baseline'
OWNER = 'Nouroz Ali'

# Reset randomness before the first run
set_seed(TRAINING_SEEDS[0])

# Point to the shared prepared data and this model's output folders
PREPARED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'enron'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'enron' / MODEL_NAME
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'enron' / MODEL_NAME

# Show the shared training seeds and batch size
print('Training seeds:', TRAINING_SEEDS)
print('Batch size:', BATCH_SIZE)

Training seeds: (42, 123, 2026)
Batch size: 64


## 2. Load Shared Prepared Data

Run `01_data_preparation.ipynb` once before this notebook. The saved preprocessing metadata controls sequence length; this notebook does not refit the vocabulary.

In [ ]:
# Load the saved splits, vocabulary, and metadata
prepared = load_prepared_data(PREPARED_DATA_DIR)

# Keep the cleaned partitions
splits = prepared['splits']

# Keep the training vocabulary
vocabulary = prepared['vocabulary']

# Keep the preprocessing configuration
metadata = prepared['metadata']

# Use the saved sequence length instead of choosing a new one
MAX_LENGTH = metadata['max_length']

# Confirm each split still has the saved number of rows
for name, expected_size in metadata['split_sizes'].items():
    assert len(splits[name]) == expected_size

# Show the loaded split sizes
pd.DataFrame([{'split': name, 'rows': len(part)} for name, part in splits.items()])

,split,rows
0,train,22524
1,validation,5632
2,test,1981


In [ ]:
# Print dataset information
print('Vocabulary size:', len(vocabulary))
print('Maximum length:', MAX_LENGTH)
print('Dataset:', metadata['dataset_id'])
print('Dataset revision:', metadata['dataset_revision'])

Vocabulary size: 10000
Maximum length: 256
Dataset: SetFit/enron_spam
Dataset revision: main


### 2.1. Encode Development Partitions

Only training and validation are encoded. Recreate the training loader for each seed; keep validation order fixed.

In [ ]:
# Encode train and validation only
encoded_datasets = {
name: EncodedTextDataset(
        texts = splits[name]['text'],
        labels = splits[name]['label'],
        vocabulary = vocabulary,
        max_length = MAX_LENGTH) for name in ('train', 'validation')
}

# Keep validation order fixed
validation_loader = make_data_loader(encoded_datasets['validation'], batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

# Inspect one validation batch
example_batch = next(iter(validation_loader))
for name, tensor in example_batch.items():
    print(name, tuple(tensor.shape))

# Recreate the training loader inside every seeded run, before training.
# Do not encode or evaluate the test partition during development.

tokens (64, 256)
lengths (64,)
labels (64,)
